<a href="https://colab.research.google.com/github/MawiyaManzar/AI-Engineering/blob/main/MedicalRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# we will use Pinecone for vector DB and integrated embeddings for easier execution.
'''
1. Langchainvectorstore.

'''

'\n1. Langchainvectorstore.\n\n'

In [2]:
!pip install langchain-text-splitters

In [3]:
!pip install langchain-pinecone

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [8]:
!pip install pinecone datasets -q

In [13]:
from pinecone import Pinecone
from datasets import load_dataset
from google.colab import userdata

In [10]:
pc = Pinecone(
    api_key=userdata.get("PINECONE_API_KEY")
)

In [11]:
index = pc.Index("medical-index")

In [15]:
dataset = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",'en',
    split="train[:100]"
)

medical_o1_sft.json:   0%|          | 0.00/58.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

In [19]:
print(dataset)
print(dataset[0])

Dataset({
    features: ['Question', 'Complex_CoT', 'Response'],
    num_rows: 100
})
{'Question': 'Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?', 'Complex_CoT': "Okay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.\n\nSo, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?\n\nOh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be the

In [20]:
print(dataset[0]["Complex_CoT"])

Okay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?

But wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.

So, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?

Oh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.

Let's put this together: if a blood clot from the leg somehow travels to the left side of the heart, it could shoot off to the brain and cause that sudden weakness by blocking blood flow there.

Hmm, but how would the clot get from the right side of the heart to the left without going through the lungs and getting filtered out?

Here's where our cardiac anomaly comes in: a patent fora

In [23]:
from pprint import pprint

In [28]:
records = []

for i, item in enumerate(dataset):

    text = f"""
    Question:
    {item['Question']}

    Reasoning:
    {item['Complex_CoT']}

    Answer:
    {item['Response']}
    """

    records.append({
        "_id":str(i),
        "text":text,
        "category":"medical"
    })

pprint(records[0]["text"][:1000])

('\n'
 '    Question:\n'
 '    Given the symptoms of sudden weakness in the left arm and leg, recent '
 'long-distance travel, and the presence of swollen and tender right lower '
 'leg, what specific cardiac abnormality is most likely to be found upon '
 'further evaluation that could explain these findings?\n'
 '\n'
 '    Reasoning:\n'
 "    Okay, let's see what's going on here. We've got sudden weakness in the "
 "person's left arm and leg - and that screams something neuro-related, maybe "
 'a stroke?\n'
 '\n'
 "But wait, there's more. The right lower leg is swollen and tender, which is "
 'like waving a big flag for deep vein thrombosis, especially after a long '
 'flight or sitting around a lot.\n'
 '\n'
 "So, now I'm thinking, how could a clot in the leg end up causing issues like "
 'weakness or stroke symptoms?\n'
 '\n'
 "Oh, right! There's this thing called a paradoxical embolism. It can happen "
 "if there's some kind of short circuit in the heart - like a hole that "
 "shou

In [29]:
batch_size = 20

for i in range(0, len(records), batch_size):

    batch = records[i:i + batch_size]

    index.upsert_records(
        namespace="medical-rag",
        records=batch
    )

    print(f"Uploaded {i + len(batch)} records")

Uploaded 20 records
Uploaded 40 records
Uploaded 60 records
Uploaded 80 records
Uploaded 100 records


In [30]:
results = index.search(
    namespace="medical-rag",
    query={
        "top_k": 3,
        "inputs": {
            "text": "treatment of hypothyroidism in ischemic heart disease"
        }
    }
)

In [31]:
for hit in results["result"]["hits"]:

    print("SCORE:", hit["_score"])
    print(hit["fields"]["text"][:1000])

    print("=" * 80)

SCORE: 0.6457106471061707

    Question:
    What is the recommended approach for initiating treatment of hypothyroidism in a patient with ischemic heart disease?

    Reasoning:
    Okay, so we've got a patient with both hypothyroidism and ischemic heart disease. That's quite a combination because we need to be really careful with how we manage the thyroid issue without putting too much strain on the heart. Hypothyroidism means they need more thyroid hormone, usually with levothyroxine. But, there's a catch. Thyroid hormones tend to speed things up - they increase heart rate and how hard the heart works. That can be a problem if the patient's heart already struggles with blood flow because of the ischemic heart disease.

Now, if we were to just give them the usual dose of thyroid hormones, we could overload the heart. That could bring about more ischemic symptoms or even trigger a heart event. Definitely not something we want to do. So the smart move here is to be super cautious. We s

In [32]:
!pip install openai -q

In [33]:
from openai import OpenAI
from google.colab import userdata

In [34]:
client = OpenAI(
    api_key=userdata.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [36]:
context = "\n\n".join([
    hit["fields"]["text"]
    for hit in results["result"]["hits"]
])

query = "What is the recommended approach for initiating treatment of hypothyroidism in ischemic heart disease?"

prompt = f"""
You are a medical assistant.

Answer ONLY using the retrieved context.

If the answer is not found in the context, say:
"I could not find the answer in the retrieved context."

Retrieved Context:
{context}

Question:
{query}
"""

In [39]:
response = client.chat.completions.create(
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2
)

In [40]:
print(response.choices[0].message.content)

Start with alow dose of levothyroxine and gradually increase the dosage while closely monitoring the patient’s response.
